A continuación te dejo el código para la segmentación de imágenes usando una arquitectura U-Net para segmentar imágenes satelitales. Puedes usar datos satelitales (GeoTIFF) y su correspondiente máscara de segmentación.

In [ ]:
# Importación de librerías
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import rasterio
import numpy as np
from torchvision import transforms

In [ ]:
# Función para leer imágenes GeoTIFF
def load_geotiff(image_path):
    with rasterio.open(image_path) as src:
        img_array = src.read([1, 2, 3])  # Leer las bandas RGB
        img_array = np.moveaxis(img_array, 0, -1)  # Cambiar el orden para que sea (height, width, channels)
    return img_array

In [ ]:
# Definir el modelo U-Net
class UNet(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNet, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, out_channels, kernel_size=3, padding=1),
            nn.ReLU()
        )

    def forward(self, x):
        x1 = self.encoder(x)
        x2 = self.decoder(x1)
        return x2

In [ ]:
# Crear una clase Dataset personalizada para GeoTIFF
class GeoTIFFDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = load_geotiff(self.image_paths[idx])
        mask = load_geotiff(self.mask_paths[idx])  # Cargar la máscara
        mask = np.where(mask > 0, 1, 0)  # Convertir la máscara a 0s y 1s

        if self.transform:
            img = self.transform(img)
            mask = self.transform(mask)

        return img, mask

In [ ]:
# Definir transformaciones
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Suponiendo que tienes listas de rutas de imágenes y máscaras
image_paths = ['path/to/image1.tif', 'path/to/image2.tif']  # Rutas a las imágenes GeoTIFF
mask_paths = ['path/to/mask1.tif', 'path/to/mask2.tif']  # Rutas a las máscaras correspondientes

In [ ]:
# Crear el DataLoader
dataset = GeoTIFFDataset(image_paths, mask_paths, transform)
data_loader = DataLoader(dataset, batch_size=1, shuffle=True)

In [ ]:
# Definir el modelo U-Net
model = UNet(in_channels=3, out_channels=1)

In [ ]:
# Definir el optimizador y la función de pérdida
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()